In [2]:
import pandas as pd

df_train = pd.read_csv('../data/application_train.csv')
print(df_train.shape)
df_train.head()

(307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df_train['TARGET'].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [4]:
missing = df_train.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_train)) * 100
missing_pct[missing_pct > 0].head(20)

COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
COMMONAREA_MEDI             69.872297
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_MODE       68.354953
FLOORSMIN_MODE              67.848630
FLOORSMIN_AVG               67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_AVG             66.497784
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_MEDI            66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MEDI               59.376738
LANDAREA_AVG                59.376738
LANDAREA_MODE               59.376738
dtype: float64

In [5]:
print(f"Total columns with missing values: {(missing_pct > 0).sum()}")
print(f"Columns with >40% missing: {(missing_pct > 40).sum()}")
print(f"Columns with >70% missing: {(missing_pct > 70).sum()}")

df_train.dtypes.value_counts()

Total columns with missing values: 67
Columns with >40% missing: 49
Columns with >70% missing: 0


float64    65
int64      41
str        16
Name: count, dtype: int64

In [6]:
# See which columns are categorical
categorical_cols = df_train.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

# Look at income and employment - key for "income stability" feature (objective 3)
df_train[['AMT_INCOME_TOTAL', 'DAYS_EMPLOYED', 'NAME_INCOME_TYPE', 'DAYS_BIRTH']].describe()

['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


/tmp/ipykernel_1359/1502898612.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_train.select_dtypes(include='object').columns.tolist()


,AMT_INCOME_TOTAL,DAYS_EMPLOYED,DAYS_BIRTH
count,3.075110e+05,307511.000000,307511.000000
mean,1.687979e+05,63815.045904,-16036.995067
std,2.371231e+05,141275.766519,4363.988632
min,2.565000e+04,-17912.000000,-25229.000000
25%,1.125000e+05,-2760.000000,-19682.000000
50%,1.471500e+05,-1213.000000,-15750.000000
75%,2.025000e+05,-289.000000,-12413.000000
max,1.170000e+08,365243.000000,-7489.000000


In [7]:
# How many applicants have this placeholder value?
anomaly_count = (df_train['DAYS_EMPLOYED'] == 365243).sum()
print(f"Applicants with DAYS_EMPLOYED = 365243: {anomaly_count} ({anomaly_count/len(df_train)*100:.1f}%)")

# Check if it correlates with a specific income type (likely pensioners/unemployed)
df_train[df_train['DAYS_EMPLOYED'] == 365243]['NAME_INCOME_TYPE'].value_counts()

Applicants with DAYS_EMPLOYED = 365243: 55374 (18.0%)


NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64

In [8]:
# Create the flag before altering the column
df_train['IS_RETIRED_OR_UNEMPLOYED'] = (df_train['DAYS_EMPLOYED'] == 365243).astype(int)

# Replace the placeholder with NaN
df_train['DAYS_EMPLOYED'] = df_train['DAYS_EMPLOYED'].replace(365243, pd.NA)

# Confirm the fix
df_train[['DAYS_EMPLOYED', 'IS_RETIRED_OR_UNEMPLOYED']].describe()

/tmp/ipykernel_1359/992546401.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train['IS_RETIRED_OR_UNEMPLOYED'] = (df_train['DAYS_EMPLOYED'] == 365243).astype(int)


,IS_RETIRED_OR_UNEMPLOYED
count,307511.000000
mean,0.180072
std,0.384248
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [9]:
# Income relative to how long they've been employed - proxy for income stability
df_train['DAYS_EMPLOYED_YEARS'] = -df_train['DAYS_EMPLOYED'] / 365  # convert to positive years

df_train['INCOME_PER_EMPLOYED_YEAR'] = df_train['AMT_INCOME_TOTAL'] / (df_train['DAYS_EMPLOYED_YEARS'] + 1)  # +1 avoids divide-by-zero

df_train[['DAYS_EMPLOYED_YEARS', 'INCOME_PER_EMPLOYED_YEAR']].describe()

/tmp/ipykernel_1359/2546688222.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train['DAYS_EMPLOYED_YEARS'] = -df_train['DAYS_EMPLOYED'] / 365  # convert to positive years
/tmp/ipykernel_1359/2546688222.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train['INCOME_PER_EMPLOYED_YEAR'] = df_train['AMT_INCOME_TOTAL'] / (df_train['DAYS_EMPLOYED_YEARS'] + 1)  # +1 avoids divide-by-zero


,DAYS_EMPLOYED_YEARS,INCOME_PER_EMPLOYED_YEAR
count,252137.000000,252137.0
unique,12573.000000,90294.0
top,0.547945,41062.5
freq,156.000000,81.0


In [10]:
import numpy as np

# Redo the fix properly
df_train['DAYS_EMPLOYED'] = df_train['DAYS_EMPLOYED'].replace(365243, np.nan)

# Recalculate the derived features
df_train['DAYS_EMPLOYED_YEARS'] = -df_train['DAYS_EMPLOYED'] / 365
df_train['INCOME_PER_EMPLOYED_YEAR'] = df_train['AMT_INCOME_TOTAL'] / (df_train['DAYS_EMPLOYED_YEARS'] + 1)

# Confirm dtypes are correct now
print(df_train[['DAYS_EMPLOYED', 'DAYS_EMPLOYED_YEARS', 'INCOME_PER_EMPLOYED_YEAR']].dtypes)
df_train[['DAYS_EMPLOYED_YEARS', 'INCOME_PER_EMPLOYED_YEAR']].describe()

DAYS_EMPLOYED               object
DAYS_EMPLOYED_YEARS         object
INCOME_PER_EMPLOYED_YEAR    object
dtype: object


,DAYS_EMPLOYED_YEARS,INCOME_PER_EMPLOYED_YEAR
count,252137.000000,252137.0
unique,12573.000000,90294.0
top,0.547945,41062.5
freq,156.000000,81.0


In [11]:
# Force DAYS_EMPLOYED back to proper numeric type
df_train['DAYS_EMPLOYED'] = pd.to_numeric(df_train['DAYS_EMPLOYED'], errors='coerce')

# Recalculate derived features
df_train['DAYS_EMPLOYED_YEARS'] = -df_train['DAYS_EMPLOYED'] / 365
df_train['INCOME_PER_EMPLOYED_YEAR'] = df_train['AMT_INCOME_TOTAL'] / (df_train['DAYS_EMPLOYED_YEARS'] + 1)

# Confirm dtypes
print(df_train[['DAYS_EMPLOYED', 'DAYS_EMPLOYED_YEARS', 'INCOME_PER_EMPLOYED_YEAR']].dtypes)
df_train[['DAYS_EMPLOYED_YEARS', 'INCOME_PER_EMPLOYED_YEAR']].describe()

DAYS_EMPLOYED               float64
DAYS_EMPLOYED_YEARS         float64
INCOME_PER_EMPLOYED_YEAR    float64
dtype: object


,DAYS_EMPLOYED_YEARS,INCOME_PER_EMPLOYED_YEAR
count,252137.000000,2.521370e+05
mean,6.531971,4.102365e+04
std,6.406466,7.941290e+04
min,-0.000000,7.911215e+02
25%,2.101370,1.522949e+04
50%,4.515068,2.849089e+04
75%,8.698630,5.284749e+04
max,49.073973,3.318182e+07


In [12]:
# How extreme is this outlier really?
print(df_train['AMT_INCOME_TOTAL'].sort_values(ascending=False).head(10))

# How many applicants are "reasonably" above the norm vs this one extreme case?
print(f"\n99th percentile: {df_train['AMT_INCOME_TOTAL'].quantile(0.99)}")
print(f"99.9th percentile: {df_train['AMT_INCOME_TOTAL'].quantile(0.999)}")

12840     117000000.0
203693     18000090.0
246858     13500000.0
77768       9000000.0
131127      6750000.0
103006      4500000.0
287463      4500000.0
187833      4500000.0
204564      4500000.0
181698      3950059.5
Name: AMT_INCOME_TOTAL, dtype: float64

99th percentile: 472500.0
99.9th percentile: 900000.0


In [13]:
df_train['AMT_INCOME_TOTAL_LOG'] = np.log1p(df_train['AMT_INCOME_TOTAL'])

df_train[['AMT_INCOME_TOTAL', 'AMT_INCOME_TOTAL_LOG']].describe()

/tmp/ipykernel_1359/1418529032.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train['AMT_INCOME_TOTAL_LOG'] = np.log1p(df_train['AMT_INCOME_TOTAL'])


,AMT_INCOME_TOTAL,AMT_INCOME_TOTAL_LOG
count,3.075110e+05,307511.000000
mean,1.687979e+05,11.909245
std,2.371231e+05,0.488906
min,2.565000e+04,10.152338
25%,1.125000e+05,11.630717
50%,1.471500e+05,11.899215
75%,2.025000e+05,12.218500
max,1.170000e+08,18.577685


In [14]:
df_installments = pd.read_csv('../data/installments_payments.csv')
print(df_installments.shape)
df_installments.head()

(13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [15]:
# Row-level behavioral signals
df_installments['PAYMENT_DELAY_DAYS'] = df_installments['DAYS_ENTRY_PAYMENT'] - df_installments['DAYS_INSTALMENT']
df_installments['PAYMENT_DIFF'] = df_installments['AMT_PAYMENT'] - df_installments['AMT_INSTALMENT']

# Aggregate to one row per applicant
installment_features = df_installments.groupby('SK_ID_CURR').agg(
    avg_payment_delay=('PAYMENT_DELAY_DAYS', 'mean'),
    std_payment_delay=('PAYMENT_DELAY_DAYS', 'std'),
    pct_late_payments=('PAYMENT_DELAY_DAYS', lambda x: (x > 0).mean()),
    avg_payment_diff=('PAYMENT_DIFF', 'mean'),
    std_payment_amount=('AMT_INSTALMENT', 'std'),
    num_installments=('SK_ID_CURR', 'count')
).reset_index()

print(installment_features.shape)
installment_features.head(10)

(339587, 7)


,SK_ID_CURR,avg_payment_delay,std_payment_delay,pct_late_payments,avg_payment_diff,std_payment_amount,num_installments
0,100001,-7.285714,14.625483,0.142857,0.000000,5076.676624,7
1,100002,-20.421053,4.925171,0.000000,0.000000,10058.037722,19
2,100003,-7.160000,3.726929,0.000000,0.000000,110542.592300,25
3,100004,-7.666667,4.163332,0.000000,0.000000,3011.871810,3
4,100005,-23.555556,13.510284,0.111111,0.000000,4281.015000,9
5,100006,-19.375000,25.397835,0.000000,0.000000,168097.624347,16
6,100007,-3.636364,7.991604,0.242424,-452.384318,7852.910669,66
7,100008,26.114286,224.764422,0.028571,-342.461571,70634.672225,35
8,100009,-8.588235,5.111463,0.019608,0.000000,3067.815701,51
9,100010,-11.900000,6.967384,0.000000,0.000000,44.910667,10


In [16]:
df_train_enriched = df_train.merge(installment_features, on='SK_ID_CURR', how='left')

print(df_train_enriched.shape)
print(f"Applicants with no installment history: {df_train_enriched['avg_payment_delay'].isnull().sum()}")

(307511, 132)
Applicants with no installment history: 15876


In [17]:
# Flag applicants with no prior loan history
df_train_enriched['NO_INSTALLMENT_HISTORY'] = df_train_enriched['avg_payment_delay'].isnull().astype(int)

# Impute the behavioral features with 0 (reasonable default: no history = no observed lateness/irregularity)
behavioral_cols = ['avg_payment_delay', 'std_payment_delay', 'pct_late_payments', 
                    'avg_payment_diff', 'std_payment_amount', 'num_installments']

df_train_enriched[behavioral_cols] = df_train_enriched[behavioral_cols].fillna(0)

# Confirm no more nulls in these columns
df_train_enriched[behavioral_cols].isnull().sum()

/tmp/ipykernel_1359/1588695001.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train_enriched['NO_INSTALLMENT_HISTORY'] = df_train_enriched['avg_payment_delay'].isnull().astype(int)


avg_payment_delay     0
std_payment_delay     0
pct_late_payments     0
avg_payment_diff      0
std_payment_amount    0
num_installments      0
dtype: int64

In [18]:
from sklearn.model_selection import train_test_split

# Select features - exclude ID, target, and raw columns we've transformed
feature_cols = [col for col in df_train_enriched.columns 
                 if col not in ['SK_ID_CURR', 'TARGET']]

X = df_train_enriched[feature_cols]
y = df_train_enriched['TARGET']

print(f"Feature count: {len(feature_cols)}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Feature count: 131
X shape: (307511, 131), y shape: (307511,)


In [19]:
# Check remaining categorical columns
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns remaining: {len(categorical_cols)}")
print(categorical_cols)

# Check remaining missing values
remaining_missing = X.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(f"\nColumns with missing values: {len(remaining_missing)}")
print(remaining_missing.head(20))

/tmp/ipykernel_1359/1914581329.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


Categorical columns remaining: 16
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

Columns with missing values: 70
COMMONAREA_MODE             214865
COMMONAREA_AVG              214865
COMMONAREA_MEDI             214865
NONLIVINGAPARTMENTS_AVG     213514
NONLIVINGAPARTMENTS_MODE    213514
NONLIVINGAPARTMENTS_MEDI    213514
FONDKAPREMONT_MODE          210295
LIVINGAPARTMENTS_AVG        210199
LIVINGAPARTMENTS_MEDI       210199
LIVINGAPARTMENTS_MODE       210199
FLOORSMIN_MODE              208642
FLOORSMIN_MEDI              208642
FLOORSMIN_AVG               208642
YEARS_BUILD_MODE            204488
YEARS_BUILD_MEDI            204488
YEARS_BUILD_AVG             204488
OWN_CAR_AGE                 202929
LANDA

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Separate numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

# Numeric pipeline: fill missing with median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: fill missing with a placeholder, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine both into one preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print("Preprocessor built successfully")

Numeric columns: 115
Categorical columns: 16
Preprocessor built successfully


/tmp/ipykernel_1359/616747410.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")

Train shape: (246008, 131)
Test shape: (61503, 131)
Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64
Test target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# Full pipeline: preprocessing + model
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Fit on training data
log_reg_pipeline.fit(X_train, y_train)

# Predict probabilities on test data
y_pred_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]
y_pred = log_reg_pipeline.predict(X_test)

# Evaluate
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

AUC-ROC: 0.7526

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.69      0.81     56538
           1       0.16      0.68      0.26      4965

    accuracy                           0.69     61503
   macro avg       0.56      0.69      0.54     61503
weighted avg       0.90      0.69      0.76     61503



In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)

y_pred_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]
y_pred_rf = rf_pipeline.predict(X_test)

auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
print(f"Random Forest AUC-ROC: {auc_rf:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest AUC-ROC: 0.7406

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.96     56538
           1       0.44      0.05      0.09      4965

    accuracy                           0.92     61503
   macro avg       0.68      0.52      0.52     61503
weighted avg       0.88      0.92      0.89     61503



In [25]:
import xgboost as xgb

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        n_estimators=100, 
        scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
        random_state=42,
        eval_metric='logloss'
    ))
])

xgb_pipeline.fit(X_train, y_train)

y_pred_proba_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_pipeline.predict(X_test)

auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
print(f"XGBoost AUC-ROC: {auc_xgb:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost AUC-ROC: 0.7560

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.75      0.84     56538
           1       0.18      0.63      0.28      4965

    accuracy                           0.74     61503
   macro avg       0.57      0.69      0.56     61503
weighted avg       0.90      0.74      0.80     61503

